In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## Крок 1: Визначте моделі Pydantic для структурованих результатів

Ці моделі визначають **схему**, яку агенти повертатимуть. Використання `response_format` з Pydantic забезпечує:
- ✅ Безпечне за типом вилучення даних
- ✅ Автоматична валідація
- ✅ Відсутність помилок розбору з вільнотекстових відповідей
- ✅ Легкий умовний роутинг на основі полів


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Крок 2: Створення інструмента для бронювання готелів

Цей інструмент саме те, що викликатиме **availability_agent** для перевірки наявності кімнат. Ми використовуємо декоратор `@ai_function` щоб:
- Перетворити функцію Python на інструмент, доступний для виклику ШІ
- Автоматично генерувати JSON-схему для LLM
- Опрацьовувати валідацію параметрів
- Дозволити автоматичний виклик агентами

Для цього демо:
- **Стокгольм, Сіетл, Токіо, Лондон, Амстердам** → Кімнати є ✅
- **Всі інші міста** → Кімнат немає ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Крок 3: Визначення функцій умов для маршрутизації

Ці функції перевіряють відповідь агента і визначають, яким шляхом йти у робочому процесі.

**Ключовий шаблон:**
1. Перевірте, чи є повідомлення `AgentExecutorResponse`
2. Проаналізуйте структурований вихід (модель Pydantic)
3. Поверніть `True` або `False`, щоб керувати маршрутизацією

Робочий процес буде оцінювати ці умови на **ребрах**, щоб вирішити, який виконавець буде викликаний наступним.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Крок 4: Створіть власний виконавець відображення

Виконавці — це компоненти робочого процесу, які виконують трансформації або побічні ефекти. Ми використовуємо декоратор `@executor` для створення власного виконавця, який відображає фінальний результат.

**Ключові поняття:**
- `@executor(id="...")` - Реєструє функцію як виконавець робочого процесу
- `WorkflowContext[Never, str]` - Типові підказки для вводу/виводу
- `ctx.yield_output(...)` - Повертає остаточний результат робочого процесу


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Крок 5: Завантаження змінних середовища

Налаштуйте клієнт LLM. Цей приклад працює з:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — рекомендується)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## Крок 6: Створення агентів ШІ зі структурованими результатами

Ми створюємо **трьох спеціалізованих агентів**, кожен з яких обгорнутий в `AgentExecutor`:

1. **availability_agent** - Перевіряє наявність готелю за допомогою інструменту
2. **alternative_agent** - Пропонує альтернативні міста (коли немає вільних номерів)
3. **booking_agent** - Заохочує бронювання (коли номери доступні)

**Основні характеристики:**
- `tools=[hotel_booking]` - Надає інструмент агенту
- `response_format=PydanticModel` - Забезпечує структурований JSON-вивід
- `AgentExecutor(..., id="...")` - Обгортає агента для використання у робочому процесі


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## Крок 7: Побудова робочого процесу з умовними ребрами

Тепер ми використовуємо `WorkflowBuilder`, щоб побудувати граф з умовною маршрутизацією:

**Структура робочого процесу:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Основні методи:**
- `.set_start_executor(...)` - Встановлює точку входу
- `.add_edge(from, to, condition=...)` - Додає умовне ребро
- `.build()` - Завершує робочий процес


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Крок 8: Запустіть тестовий випадок 1 - Місто БЕЗ доступності (Париж)

Давайте перевіримо шлях **без доступності**, запросивши готелі в Парижі (де немає номерів у нашій симуляції).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Крок 9: Запустіть тестовий випадок 2 - Місто З наявністю (Стокгольм)

Тепер перевіримо шлях **наявності**, зробивши запит на готелі в Стокгольмі (у якому є кімнати в нашій симуляції).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Основні висновки та подальші кроки

### ✅ Чому ви навчились:

1. **Патерн WorkflowBuilder**
   - Використовуйте `.set_start_executor()` для визначення точки входу
   - Використовуйте `.add_edge(from, to, condition=...)` для умовної маршрутизації
   - Викликайте `.build()` для завершення побудови workflow

2. **Умовна маршрутизація**
   - Функції умов переглядають `AgentExecutorResponse`
   - Аналізуйте структуровані вихідні дані для прийняття рішень про маршрут
   - Поверніть `True` щоб активувати ребро, `False` щоб його пропустити

3. **Інтеграція інструментів**
   - Використовуйте `@ai_function` для перетворення Python функцій у AI інструменти
   - Агенти автоматично викликають інструменти за потреби
   - Інструменти повертають JSON, який агенти можуть аналізувати

4. **Структуровані вихідні дані**
   - Використовуйте моделі Pydantic для типобезпечного вилучення даних
   - Встановлюйте `response_format=MyModel` при створенні агентів
   - Аналізуйте відповіді за допомогою `Model.model_validate_json()`

5. **Користувацькі виконавці**
   - Використовуйте `@executor(id="...")` для створення компонентів workflow
   - Виконавці можуть трансформувати дані або виконувати побічні ефекти
   - Використовуйте `ctx.yield_output()` для виведення результатів workflow

### 🚀 Застосування в реальному світі:

- **Бронювання подорожей**: Перевірка доступності, пропозиція альтернатив, порівняння варіантів
- **Обслуговування клієнтів**: Маршрутизація за типом проблеми, настроєм, пріоритетом
- **Електронна комерція**: Перевірка наявності, пропозиція альтернатив, обробка замовлень
- **Модерація контенту**: Маршрутизація за рівнями токсичності, позначками користувачів
- **Процеси затвердження**: Маршрутизація за сумою, роллю користувача, рівнем ризику
- **Багатоступеневі процеси**: Маршрутизація за якістю та повнотою даних

### 📚 Наступні кроки:

- Додати більш складні умови (декілька критеріїв)
- Реалізувати цикли з управлінням станом workflow
- Додати підпроцеси для повторно використовуваних компонентів
- Інтегрувати з реальними API (бронювання готелів, системи інвентаризації)
- Додати обробку помилок та запасні варіанти
- Візуалізувати workflow за допомогою вбудованих інструментів візуалізації


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
